In [12]:
import os
import sys
import subprocess

N_JOBS = 2
os.environ['OMP_NUM_THREADS'] = str(N_JOBS)
os.environ['MKL_NUM_THREADS'] = str(N_JOBS)
os.environ['OPENBLAS_NUM_THREADS'] = str(N_JOBS)
os.environ['NUMEXPR_NUM_THREADS'] = str(N_JOBS)
os.environ['NUMBA_NUM_THREADS'] = str(N_JOBS)
os.environ.setdefault('NUMBA_CACHE_DIR', os.path.join(os.path.expanduser('~'), '.cache', 'numba'))
os.makedirs(os.environ['NUMBA_CACHE_DIR'], exist_ok=True)


def _patch_numba_coverage():
    """numba 0.63 ждёт coverage.types.Tracer, в coverage 7.6 его нет."""
    try:
        from typing import Any, Callable
        import coverage.types as ct
        if not hasattr(ct, 'Tracer'):
            ct.Tracer = ct.TracerCore
        if not hasattr(ct, 'TShouldTraceFn'):
            ct.TShouldTraceFn = Callable[..., Any]
        if not hasattr(ct, 'TShouldStartContextFn'):
            ct.TShouldStartContextFn = Callable[..., Any]
    except Exception:
        pass


_patch_numba_coverage()

for _pkg in ('lightgbm', 'catboost'):
    try:
        __import__(_pkg)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', _pkg])

In [13]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_regression

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)


def comma_to_float(s):
    return pd.to_numeric(
        pd.Series(s, copy=False).astype(str).str.replace(',', '.', regex=False),
        errors='coerce',
    )


def load_bgg(path):
    d = pd.read_csv(path)
    if 'Rating Average' in d.columns:
        d['Rating Average'] = comma_to_float(d['Rating Average'])
    d['Complexity Average'] = comma_to_float(d['Complexity Average'])
    d['Mechanics'] = d['Mechanics'].fillna('Unknown')
    d['Domains'] = d['Domains'].fillna('Unknown')
    return d

In [14]:
df = load_bgg('train_data.csv')
# Датасет уже отфильтрован: Users Rated min = 30
print('Users Rated min:', df['Users Rated'].min())
df.info()

Users Rated min: 30
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15257 entries, 0 to 15256
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ID                  15247 non-null  float64
 1   Name                15257 non-null  object 
 2   Year Published      15256 non-null  float64
 3   Min Players         15257 non-null  int64  
 4   Max Players         15257 non-null  int64  
 5   Play Time           15257 non-null  int64  
 6   Min Age             15257 non-null  int64  
 7   Users Rated         15257 non-null  int64  
 8   Rating Average      15257 non-null  float64
 9   BGG Rank            15257 non-null  int64  
 10  Complexity Average  15257 non-null  float64
 11  Owned Users         15240 non-null  float64
 12  Mechanics           15257 non-null  object 
 13  Domains             15257 non-null  object 
dtypes: float64(5), int64(6), object(3)
memory usage: 1.6+ MB


In [15]:
df.describe()

,ID,Year Published,Min Players,Max Players,Play Time,Min Age,Users Rated,Rating Average,BGG Rank,Complexity Average,Owned Users
count,15247.000000,15256.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15240.000000
mean,108221.590214,1984.144140,2.021236,5.642918,92.801337,9.581766,819.530773,6.399775,10186.327784,1.991064,1372.209843
std,98862.566448,211.725679,0.696323,12.551944,607.611378,3.671560,3286.396397,0.936488,5863.744570,0.848816,4681.296849
min,1.000000,-3500.000000,0.000000,0.000000,0.000000,0.000000,30.000000,1.050000,1.000000,0.000000,0.000000
25%,10907.000000,2001.000000,2.000000,4.000000,30.000000,8.000000,55.000000,5.820000,5103.000000,1.330000,146.000000
50%,88316.000000,2011.000000,2.000000,4.000000,45.000000,10.000000,119.000000,6.430000,10171.000000,2.000000,308.000000
75%,193045.500000,2016.000000,2.000000,6.000000,90.000000,12.000000,379.000000,7.020000,15280.000000,2.540000,856.250000
max,331787.000000,2022.000000,10.000000,999.000000,60000.000000,25.000000,102214.000000,9.580000,20344.000000,5.000000,155312.000000


In [16]:
df.head()
df['Mechanics'] = df['Mechanics'].fillna('Unknown')
df['Domains'] = df['Domains'].fillna('Unknown')
df = df.drop(['ID'], axis=1)
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15257 entries, 0 to 15256
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Name                15257 non-null  object 
 1   Year Published      15256 non-null  float64
 2   Min Players         15257 non-null  int64  
 3   Max Players         15257 non-null  int64  
 4   Play Time           15257 non-null  int64  
 5   Min Age             15257 non-null  int64  
 6   Users Rated         15257 non-null  int64  
 7   Rating Average      15257 non-null  float64
 8   BGG Rank            15257 non-null  int64  
 9   Complexity Average  15257 non-null  float64
 10  Owned Users         15240 non-null  float64
 11  Mechanics           15257 non-null  object 
 12  Domains             15257 non-null  object 
dtypes: float64(4), int64(6), object(3)
memory usage: 1.5+ MB


In [17]:
df.select_dtypes(include = ['float64', 'int64']).corr()

,Year Published,Min Players,Max Players,Play Time,Min Age,Users Rated,Rating Average,BGG Rank,Complexity Average,Owned Users
Year Published,1.000000,-0.016228,-0.007565,0.007232,0.091842,0.012105,0.079546,-0.085764,0.036625,0.018334
Min Players,-0.016228,1.000000,0.108232,0.028361,0.042317,-0.016869,-0.158291,0.096783,-0.175218,-0.028768
Max Players,-0.007565,0.108232,1.000000,-0.007540,0.017650,0.002174,-0.043786,0.028195,-0.105990,0.003990
Play Time,0.007232,0.028361,-0.007540,1.000000,0.038525,-0.003893,0.080006,-0.041497,0.168908,-0.002098
Min Age,0.091842,0.042317,0.017650,0.038525,1.000000,0.075521,0.187403,-0.210580,0.281572,0.086086
Users Rated,0.012105,-0.016869,0.002174,-0.003893,0.075521,1.000000,0.175867,-0.321523,0.094232,0.984080
Rating Average,0.079546,-0.158291,-0.043786,0.080006,0.187403,0.175867,1.000000,-0.737897,0.470200,0.184917
BGG Rank,-0.085764,0.096783,0.028195,-0.041497,-0.210580,-0.321523,-0.737897,1.000000,-0.378614,-0.342471
Complexity Average,0.036625,-0.175218,-0.105990,0.168908,0.281572,0.094232,0.470200,-0.378614,1.000000,0.096454
Owned Users,0.018334,-0.028768,0.003990,-0.002098,0.086086,0.984080,0.184917,-0.342471,0.096454,1.000000


In [18]:
domain_dummies = df['Domains'].str.replace(r'\s*,\s*', ',', regex = True).str.get_dummies(sep = ',')
domain_dummies = domain_dummies.add_prefix('Domain_')

mechanics_dummies = df['Mechanics'].str.replace(r'\s*,\s*', ',', regex = True).str.get_dummies(sep = ',')
mechanics_dummies = mechanics_dummies.loc[:, mechanics_dummies.sum() > 100]
mechanics_dummies = mechanics_dummies.add_prefix('Mech_')

df_encoded = pd.concat([df, domain_dummies, mechanics_dummies], axis=1)

In [19]:
df_encoded = df_encoded.drop(['Name', "Mechanics", "Domains"], axis = 1)


In [20]:
X_mi = df_encoded.drop(['Rating Average'], axis=1)
X_mi = X_mi.fillna(X_mi.median(numeric_only=True))
print(df_encoded.info())
mi_scores = mutual_info_regression(X_mi, df_encoded['Rating Average'], discrete_features='auto', random_state=42)
cols = X_mi.columns
    
mi_df = pd.DataFrame({
    'Feature': cols,
    'MI_Score': mi_scores
}).sort_values(by='MI_Score', ascending=False).reset_index(drop=True)

with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
    print(mi_df)

mi_df.to_csv('mi_scores.csv', index=False, encoding='utf-8-sig')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15257 entries, 0 to 15256
Data columns (total 77 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   Year Published                           15256 non-null  float64
 1   Min Players                              15257 non-null  int64  
 2   Max Players                              15257 non-null  int64  
 3   Play Time                                15257 non-null  int64  
 4   Min Age                                  15257 non-null  int64  
 5   Users Rated                              15257 non-null  int64  
 6   Rating Average                           15257 non-null  float64
 7   BGG Rank                                 15257 non-null  int64  
 8   Complexity Average                       15257 non-null  float64
 9   Owned Users                              15240 non-null  float64
 10  Domain_Abstract Games                    15257

BGG Rank оставляем: колонка есть и в test, с рейтингом корреляция −0.74, ранги train/test не пересекаются.

Фичи как в сильных решениях: сырой и log rank, флаги Roll/Spin и доменов, TF-IDF механик + SVD (10), сглаженный target encoding механик/доменов (prior 80) и семьи по очищенному названию, бины игроков и длины партии, `vote_shrink = n/(n+50)`. UMAP убран. Обучение без веса по Users Rated — метрика невзвешенный RMSE.


In [21]:
N_JOBS = int(globals().get('N_JOBS', 2))

from collections import Counter, defaultdict
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.linear_model import Ridge
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from lightgbm import LGBMRegressor, early_stopping
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

NUM_RAW = [
    'Year Published', 'Min Players', 'Max Players', 'Play Time',
    'Min Age', 'Users Rated', 'Complexity Average', 'Owned Users', 'BGG Rank',
]
ZERO_AS_NA = ['Year Published', 'Play Time', 'Min Age', 'Complexity Average']
NAME_RE = r'Expanded|Edition|Remastered|Legacy|Deluxe|Reprint|2nd|Second Edition'
GOOD_MECHS = (
    'Variable Player Powers',
    'Card Drafting',
    'Worker Placement',
    'Deck Bag and Pool Building',
    'Cooperative Game',
    'Modular Board',
)
TAG_SMOOTH = 80.0
FAMILY_SMOOTH = 8.0
NAME_TFIDF = dict(ngram_range=(1, 2), min_df=5, max_features=3000)
NAME_SVD_COMPS = 16
MECH_SVD_COMPS = 10
TE_COLS = [
    'mean_mech_target', 'max_mech_target', 'min_mech_target', 'count_mech_target', 'var_mech_target',
    'mean_dom_target', 'max_dom_target', 'min_dom_target', 'count_dom_target', 'var_dom_target',
    'family_target', 'family_count',
]


def _comma_tokens(text):
    return [t.strip() for t in text.split(',') if t.strip()]


def _norm_tags(s):
    return s.fillna('Unknown').str.replace(r'\s*,\s*', ',', regex=True)


def _mech_doc(text):
    parts = []
    for t in str(text).split(','):
        tok = '_'.join(t.strip().lower().split())
        if tok:
            parts.append(tok)
    return ' '.join(parts)


def _family_keys(names):
    s = pd.Series(names, copy=False).fillna('').astype(str).str.lower()
    s = s.str.replace(r'\([^)]*\)', ' ', regex=True)
    s = s.str.replace(r':.*', ' ', regex=True)
    s = s.str.replace(
        r'\b(\d+(st|nd|rd|th)|second|third|fourth|edition|remastered|deluxe|reprint|expanded|legacy|revised)\b',
        ' ',
        regex=True,
    )
    s = s.str.replace(r'[^a-z0-9]+', ' ', regex=True).str.strip()
    s = s.mask(s.eq(''), 'unknown')
    return s.to_numpy()


def _tag_means(lists, y, smooth=TAG_SMOOTH):
    y = np.asarray(y, dtype=float)
    gmean = float(np.mean(y))
    sums, counts, sq = defaultdict(float), defaultdict(int), defaultdict(float)
    for tags, yi in zip(lists, y):
        for t in tags:
            sums[t] += yi
            sq[t] += yi * yi
            counts[t] += 1
    means = {t: (sums[t] + smooth * gmean) / (counts[t] + smooth) for t in counts}
    tag_var = {}
    for t, c in counts.items():
        if c > 1:
            tag_var[t] = max(sq[t] / c - (sums[t] / c) ** 2, 0.0)
        else:
            tag_var[t] = 0.0
    return means, gmean, counts, tag_var


def _tag_stats(lists, means, gmean, counts, tag_var):
    out = np.empty((len(lists), 5), dtype=float)
    for i, tags in enumerate(lists):
        known = [t for t in tags if t in means]
        if not known:
            out[i] = [gmean, gmean, gmean, 0.0, 0.0]
            continue
        vals = [means[t] for t in known]
        out[i, 0] = np.mean(vals)
        out[i, 1] = np.max(vals)
        out[i, 2] = np.min(vals)
        out[i, 3] = float(np.mean([counts.get(t, 0) for t in known]))
        out[i, 4] = float(np.mean([tag_var.get(t, 0.0) for t in known]))
    return out


def _family_stats(keys, means, gmean, counts):
    out = np.empty((len(keys), 2), dtype=float)
    for i, key in enumerate(keys):
        if key in means:
            out[i, 0] = means[key]
            out[i, 1] = float(counts.get(key, 0))
        else:
            out[i, 0] = gmean
            out[i, 1] = 0.0
    return out


class BoardGameFeatures(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        mech_max_features=150,
        dom_max_features=50,
        min_df=10,
        name_svd_components=NAME_SVD_COMPS,
        mech_svd_components=MECH_SVD_COMPS,
    ):
        self.mech_max_features = mech_max_features
        self.dom_max_features = dom_max_features
        self.min_df = min_df
        self.name_svd_components = name_svd_components
        self.mech_svd_components = mech_svd_components

    def _as_df(self, X):
        if isinstance(X, pd.DataFrame):
            return X.copy()
        return pd.DataFrame(X, columns=self.feature_names_in_)

    def _engineer_num(self, X):
        n = X[NUM_RAW].apply(pd.to_numeric, errors='coerce')
        swap = n['Min Players'] > n['Max Players']
        n.loc[swap, ['Min Players', 'Max Players']] = (
            n.loc[swap, ['Max Players', 'Min Players']].to_numpy()
        )

        year = n['Year Published']
        n['year_cent'] = (year - 2000) / 20
        n['is_old'] = ((year < 1990) & (year != 0) & year.notna()).astype(float)
        n['is_ancient'] = ((year < 1900) & (year != 0) & year.notna()).astype(float)
        n['is_modern'] = (year >= 2015).astype(float)
        n['is_year_future_or_weird'] = ((year < 0) | (year > 2026)).astype(float)
        year_missing = ((year == 0) | year.isna()).astype(float)
        complexity_missing = ((n['Complexity Average'] == 0) | n['Complexity Average'].isna()).astype(float)

        n['owned_per_rated'] = n['Owned Users'] / (n['Users Rated'] + 1)
        n['age_per_complexity'] = n['Min Age'] / (n['Complexity Average'] + 0.1)
        n['time_per_min_player'] = n['Play Time'] / (n['Min Players'].clip(lower=0) + 0.1)
        n['is_filler'] = ((n['Play Time'] > 0) & (n['Play Time'] <= 30)).astype(float)
        n['is_long'] = (n['Play Time'] >= 120).astype(float)
        n['player_group'] = np.select(
            [n['Min Players'] <= 1, n['Min Players'] == 2, n['Min Players'] >= 5],
            [0.0, 1.0, 3.0],
            default=2.0,
        ).astype(float)

        for c in ZERO_AS_NA:
            n.loc[n[c] == 0, c] = np.nan

        n['year_missing'] = year_missing
        n['complexity_missing'] = complexity_missing
        n['is_solo'] = (n['Min Players'] <= 1).astype(float)
        n['is_duel'] = ((n['Min Players'] == 2) & (n['Max Players'] == 2)).astype(float)
        n['is_party'] = (n['Max Players'] >= 7).astype(float)
        n['player_range'] = (n['Max Players'] - n['Min Players']).clip(lower=0)
        n['is_remastered'] = X['is_remastered'].astype(float)
        n['log_users'] = np.log1p(n['Users Rated'].clip(lower=0))
        n['log_owned'] = np.log1p(n['Owned Users'].clip(lower=0))
        n['log_time'] = np.log1p(n['Play Time'].clip(lower=0))
        n['log_rank'] = np.log1p(n['BGG Rank'].clip(lower=0))
        n['log_users_per_owned'] = n['log_users'] - n['log_owned']
        n['vote_shrink'] = n['Users Rated'].clip(lower=0) / (n['Users Rated'].clip(lower=0) + 50)
        n['complexity_x_shrink'] = n['Complexity Average'] * n['vote_shrink']
        n['log_rank_x_shrink'] = n['log_rank'] * n['vote_shrink']
        n['complexity_sq'] = n['Complexity Average'] ** 2
        n['complexity_x_log_time'] = n['Complexity Average'] * n['log_time']
        n['playtime_per_player'] = n['log_time'] / (n['Max Players'].clip(lower=1) + 0.1)

        mech = X['Mechanics'].fillna('')
        dom = X['Domains'].fillna('')
        n['has_roll_move'] = mech.str.contains('Roll / Spin and Move', regex=False).astype(float)
        n['good_mech_n'] = sum(mech.str.contains(g, regex=False).astype(float) for g in GOOD_MECHS)
        n['has_child_domain'] = dom.str.contains("Children's Games", regex=False).astype(float)
        n['has_strategy'] = dom.str.contains('Strategy Games', regex=False).astype(float)
        n['has_wargame'] = dom.str.contains('Wargames', regex=False).astype(float)
        mech_txt = _norm_tags(X['Mechanics'])
        dom_txt = _norm_tags(X['Domains'])
        n['mech_n_tags'] = mech_txt.apply(lambda s: len(_comma_tokens(s)))
        n['dom_n_tags'] = dom_txt.apply(lambda s: len(_comma_tokens(s)))
        return n

    def _fit_dom(self, X):
        dom_lists = _norm_tags(X['Domains']).map(_comma_tokens)
        dom_counts = Counter(t for tags in dom_lists for t in tags)
        self.dom_keep_ = {
            lab for lab, n in dom_counts.most_common(self.dom_max_features)
            if n >= 1
        }
        self.mlb_dom_ = MultiLabelBinarizer(classes=sorted(self.dom_keep_))
        self.mlb_dom_.fit(dom_lists.map(lambda xs: [t for t in xs if t in self.dom_keep_]))
        self.dom_cols_ = [f'Domain_{c}' for c in self.mlb_dom_.classes_]
        return dom_lists

    def _dom_frame(self, X):
        dom_lists = _norm_tags(X['Domains']).map(
            lambda s: [t for t in _comma_tokens(s) if t in self.dom_keep_]
        )
        return pd.DataFrame(
            self.mlb_dom_.transform(dom_lists),
            columns=self.dom_cols_,
            index=X.index,
        )

    def _num_frame(self, X):
        return pd.DataFrame(
            self.num_imputer_.transform(self._engineer_num(X)),
            columns=self.num_cols_,
            index=X.index,
        )

    def fit(self, X, y=None):
        X = self._as_df(X)
        y = np.asarray(y, dtype=float)
        self.feature_names_in_ = np.asarray(X.columns)

        num = self._engineer_num(X)
        self.num_cols_ = list(num.columns)
        self.num_imputer_ = SimpleImputer(strategy='median', keep_empty_features=True)
        self.num_imputer_.fit(num)

        mech_lists = _norm_tags(X['Mechanics']).map(_comma_tokens)
        dom_lists = self._fit_dom(X)
        self.mech_means_, self.mech_gmean_, self.mech_counts_, self.mech_var_ = _tag_means(mech_lists, y)
        self.dom_means_, self.dom_gmean_, self.dom_counts_, self.dom_var_ = _tag_means(dom_lists, y)
        family = _family_keys(X['Name'])
        fam_lists = [[k] for k in family]
        self.fam_means_, self.fam_gmean_, self.fam_counts_, _ = _tag_means(fam_lists, y, smooth=FAMILY_SMOOTH)

        names = X['Name'].fillna('').astype(str)
        self.tfidf_name_ = TfidfVectorizer(**NAME_TFIDF)
        tf = self.tfidf_name_.fit_transform(names)
        self.name_ridge_ = Ridge(alpha=10)
        self.name_ridge_.fit(tf, y)
        n_svd = min(self.name_svd_components, max(tf.shape[1] - 1, 1), max(tf.shape[0] - 1, 1))
        self.name_svd_ = TruncatedSVD(n_components=n_svd, random_state=42)
        self.name_svd_.fit(tf)
        self.name_svd_cols_ = [f'name_svd_{i}' for i in range(n_svd)]

        mech_docs = X['Mechanics'].fillna('').map(_mech_doc)
        self.tfidf_mech_ = TfidfVectorizer(min_df=self.min_df, token_pattern=r'[^ ]+')
        mech_tf = self.tfidf_mech_.fit_transform(mech_docs)
        n_mech = min(self.mech_svd_components, max(mech_tf.shape[1] - 1, 1), max(mech_tf.shape[0] - 1, 1))
        self.mech_svd_ = TruncatedSVD(n_components=n_mech, random_state=42)
        self.mech_svd_.fit(mech_tf)
        self.mech_svd_cols_ = [f'mech_svd_{i}' for i in range(n_mech)]
        return self

    def _pack(self, X, mech_te, dom_te, fam_te, name_pred):
        num = self._num_frame(X)
        te = pd.DataFrame(
            np.hstack([mech_te, dom_te, fam_te]),
            columns=TE_COLS,
            index=X.index,
        )
        dom = self._dom_frame(X)
        name_tf = self.tfidf_name_.transform(X['Name'].fillna('').astype(str))
        name_svd = self.name_svd_.transform(name_tf)
        svd_df = pd.DataFrame(name_svd, columns=self.name_svd_cols_, index=X.index)
        mech_tf = self.tfidf_mech_.transform(X['Mechanics'].fillna('').map(_mech_doc))
        mech_svd = self.mech_svd_.transform(mech_tf)
        mech_df = pd.DataFrame(mech_svd, columns=self.mech_svd_cols_, index=X.index)
        name_df = pd.DataFrame({'oof_name_ridge': name_pred}, index=X.index)
        out = pd.concat([num, te, dom, mech_df, svd_df, name_df], axis=1)
        return out.replace([np.inf, -np.inf], np.nan).fillna(0).to_numpy()

    def transform(self, X):
        X = self._as_df(X)
        mech_lists = _norm_tags(X['Mechanics']).map(_comma_tokens)
        dom_lists = _norm_tags(X['Domains']).map(_comma_tokens)
        mech_te = _tag_stats(mech_lists, self.mech_means_, self.mech_gmean_, self.mech_counts_, self.mech_var_)
        dom_te = _tag_stats(dom_lists, self.dom_means_, self.dom_gmean_, self.dom_counts_, self.dom_var_)
        fam_te = _family_stats(_family_keys(X['Name']), self.fam_means_, self.fam_gmean_, self.fam_counts_)
        name_pred = self.name_ridge_.predict(self.tfidf_name_.transform(X['Name'].fillna('').astype(str)))
        return self._pack(X, mech_te, dom_te, fam_te, name_pred)

    def fit_transform(self, X, y=None):
        X = self._as_df(X)
        y = np.asarray(y, dtype=float)
        self.fit(X, y)

        mech_lists = _norm_tags(X['Mechanics']).map(_comma_tokens).reset_index(drop=True)
        dom_lists = _norm_tags(X['Domains']).map(_comma_tokens).reset_index(drop=True)
        family = _family_keys(X['Name'])
        names = X['Name'].fillna('').astype(str).to_numpy()
        n = len(X)
        mech_oof = np.zeros((n, 5))
        dom_oof = np.zeros((n, 5))
        fam_oof = np.zeros((n, 2))
        name_oof = np.zeros(n)
        kf = KFold(n_splits=3, shuffle=True, random_state=42)
        for tr, va in kf.split(np.arange(n)):
            m_means, m_g, m_c, m_v = _tag_means(mech_lists.iloc[tr], y[tr])
            d_means, d_g, d_c, d_v = _tag_means(dom_lists.iloc[tr], y[tr])
            f_means, f_g, f_c, _ = _tag_means([[k] for k in family[tr]], y[tr], smooth=FAMILY_SMOOTH)
            mech_oof[va] = _tag_stats(mech_lists.iloc[va], m_means, m_g, m_c, m_v)
            dom_oof[va] = _tag_stats(dom_lists.iloc[va], d_means, d_g, d_c, d_v)
            fam_oof[va] = _family_stats(family[va], f_means, f_g, f_c)
            vec = TfidfVectorizer(**NAME_TFIDF)
            ridge = Ridge(alpha=10)
            ridge.fit(vec.fit_transform(names[tr]), y[tr])
            name_oof[va] = ridge.predict(vec.transform(names[va]))
        return self._pack(X, mech_oof, dom_oof, fam_oof, name_oof)


train = load_bgg('train_data.csv')
train['is_remastered'] = train['Name'].str.contains(NAME_RE, case=False, na=False).astype(int)

features = NUM_RAW + ['Name', 'Domains', 'Mechanics', 'is_remastered']
X = train[features]
y = train['Rating Average']

feat_kwargs = dict(
    dom_max_features=50,
    min_df=10,
    name_svd_components=NAME_SVD_COMPS,
    mech_svd_components=MECH_SVD_COMPS,
)


def make_lgbm(n_estimators=2000, **kwargs):
    params = dict(
        objective='regression',
        n_estimators=n_estimators,
        learning_rate=0.02,
        num_leaves=31,
        min_child_samples=30,
        subsample=0.85,
        colsample_bytree=0.7,
        reg_lambda=1.0,
        random_state=42,
        verbosity=-1,
        n_jobs=N_JOBS,
    )
    params.update(kwargs)
    return LGBMRegressor(**params)


def make_catboost(iterations=2000):
    return CatBoostRegressor(
        iterations=iterations,
        learning_rate=0.03,
        depth=6,
        l2_leaf_reg=3,
        random_seed=42,
        thread_count=N_JOBS,
        verbose=False,
        allow_writing_files=False,
    )


def report(name, y_true, y_pred):
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f'{name}: RMSE={rmse:.4f} MAE={mae:.4f} R2={r2:.4f}')
    return rmse, mae, r2


cv = KFold(n_splits=5, shuffle=True, random_state=42)

print('RandomizedSearch LightGBM (rank + mech SVD, без UMAP)...')
feat_gs = BoardGameFeatures(**feat_kwargs)
X_gs = feat_gs.fit_transform(X, y)
lgbm_search = make_lgbm(n_estimators=350)
param_dist = {
    'num_leaves': [15, 31, 47],
    'min_child_samples': [20, 30, 50],
    'colsample_bytree': [0.5, 0.7, 0.9],
    'learning_rate': [0.015, 0.02, 0.03],
    'reg_lambda': [0.5, 1.0, 3.0],
    'subsample': [0.7, 0.85],
}
search = RandomizedSearchCV(
    lgbm_search,
    param_distributions=param_dist,
    n_iter=6,
    cv=3,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=1,
    refit=True,
)
search.fit(X_gs, y)
best_lgbm_params = dict(search.best_params_)
print('best LightGBM params:', best_lgbm_params)
print('best CV RMSE:', -search.best_score_)

lgbm_rmse, lgbm_mae, lgbm_r2, lgbm_best_iters = [], [], [], []
cb_rmse, cb_mae, cb_r2, cb_best_iters = [], [], [], []
blend_rmse, blend_mae, blend_r2 = [], [], []
oof_lgb = np.zeros(len(X))
oof_cb = np.zeros(len(X))

for fold, (tr, va) in enumerate(cv.split(X), 1):
    feat = BoardGameFeatures(**feat_kwargs)
    Xtr = feat.fit_transform(X.iloc[tr], y.iloc[tr])
    Xva = feat.transform(X.iloc[va])
    ytr, yva = y.iloc[tr], y.iloc[va]

    lgbm = make_lgbm(**best_lgbm_params)
    lgbm.fit(
        Xtr, ytr,
        eval_X=Xva,
        eval_y=yva,
        callbacks=[early_stopping(100, verbose=False)],
    )
    p_lgb = lgbm.predict(Xva)

    cb = make_catboost()
    cb.fit(
        Xtr, ytr,
        eval_set=(Xva, yva),
        early_stopping_rounds=100,
        use_best_model=True,
    )
    p_cb = cb.predict(Xva)
    oof_lgb[va] = p_lgb
    oof_cb[va] = p_cb

    r = report(f'LightGBM fold {fold}', yva, p_lgb)
    lgbm_rmse.append(r[0]); lgbm_mae.append(r[1]); lgbm_r2.append(r[2])
    lgbm_best_iters.append(lgbm.best_iteration_)
    r = report(f'CatBoost fold {fold}', yva, p_cb)
    cb_rmse.append(r[0]); cb_mae.append(r[1]); cb_r2.append(r[2])
    cb_best_iters.append(cb.get_best_iteration() or cb.tree_count_)
    print(f'  iters LGB={lgbm_best_iters[-1]} CB={cb_best_iters[-1]}')


def summarize(title, rmse, mae, r2, iters=None):
    print()
    print(title)
    print('RMSE по фолдам:', np.array(rmse))
    print('Средний RMSE:  ', float(np.mean(rmse)))
    print('MAE по фолдам: ', np.array(mae))
    print('Средний MAE:   ', float(np.mean(mae)))
    print('R² по фолдам:  ', np.array(r2))
    print('Средний R²:    ', float(np.mean(r2)))
    if iters is not None:
        print('best_iteration:', iters, 'mean', float(np.mean(iters)))


blend_alpha = 0.6
blend_best = np.inf
for a in (0.4, 0.5, 0.6, 0.7, 0.8):
    pred_a = a * oof_lgb + (1.0 - a) * oof_cb
    rmse_a = mean_squared_error(y, pred_a) ** 0.5
    print(f'blend alpha={a:.1f} LGB: RMSE={rmse_a:.4f}')
    if rmse_a < blend_best:
        blend_best = rmse_a
        blend_alpha = a
print('chosen blend_alpha', blend_alpha)

p_blend_oof = blend_alpha * oof_lgb + (1.0 - blend_alpha) * oof_cb
for fold, (_, va) in enumerate(cv.split(X), 1):
    r = report(f'Blend fold {fold}', y.iloc[va], p_blend_oof[va])
    blend_rmse.append(r[0]); blend_mae.append(r[1]); blend_r2.append(r[2])

summarize('LightGBM', lgbm_rmse, lgbm_mae, lgbm_r2, lgbm_best_iters)
summarize('CatBoost', cb_rmse, cb_mae, cb_r2, cb_best_iters)
summarize(f'Blend {blend_alpha:.1f} LGB + {1-blend_alpha:.1f} CB', blend_rmse, blend_mae, blend_r2)

years = train['Year Published'].fillna(0)
time_tr = years < 2018
time_va = years >= 2018
print()
print('Time-based holdout: train year<2018, val year>=2018')
print('n_train', int(time_tr.sum()), 'n_val', int(time_va.sum()))

feat_time = BoardGameFeatures(**feat_kwargs)
Xtr_t = feat_time.fit_transform(X.loc[time_tr], y.loc[time_tr])
Xva_t = feat_time.transform(X.loc[time_va])
ytr_t, yva_t = y.loc[time_tr], y.loc[time_va]

lgbm_t = make_lgbm(**best_lgbm_params)
lgbm_t.fit(
    Xtr_t, ytr_t,
    eval_X=Xva_t,
    eval_y=yva_t,
    callbacks=[early_stopping(100, verbose=False)],
)
cb_t = make_catboost()
cb_t.fit(
    Xtr_t, ytr_t,
    eval_set=(Xva_t, yva_t),
    early_stopping_rounds=100,
    use_best_model=True,
)
p_lgb_t = lgbm_t.predict(Xva_t)
p_cb_t = cb_t.predict(Xva_t)
report('TimeCV LightGBM', yva_t, p_lgb_t)
report('TimeCV CatBoost', yva_t, p_cb_t)
report(f'TimeCV Blend {blend_alpha:.1f}/{1-blend_alpha:.1f}', yva_t, blend_alpha * p_lgb_t + (1.0 - blend_alpha) * p_cb_t)


RandomizedSearch LightGBM (rank + mech SVD, без UMAP)...
best LightGBM params: {'subsample': 0.7, 'reg_lambda': 0.5, 'num_leaves': 31, 'min_child_samples': 50, 'learning_rate': 0.02, 'colsample_bytree': 0.9}
best CV RMSE: 0.6929311645603425
LightGBM fold 1: RMSE=0.2128 MAE=0.1180 R2=0.9502
CatBoost fold 1: RMSE=0.2115 MAE=0.1163 R2=0.9508
  iters LGB=1994 CB=1997
LightGBM fold 2: RMSE=0.2178 MAE=0.1231 R2=0.9448
CatBoost fold 2: RMSE=0.2105 MAE=0.1160 R2=0.9485
  iters LGB=1184 CB=1987
LightGBM fold 3: RMSE=0.2182 MAE=0.1204 R2=0.9454
CatBoost fold 3: RMSE=0.2171 MAE=0.1169 R2=0.9460
  iters LGB=1326 CB=1999
LightGBM fold 4: RMSE=0.1964 MAE=0.1186 R2=0.9554
CatBoost fold 4: RMSE=0.1888 MAE=0.1152 R2=0.9588
  iters LGB=863 CB=1993
LightGBM fold 5: RMSE=0.2206 MAE=0.1242 R2=0.9446
CatBoost fold 5: RMSE=0.2145 MAE=0.1214 R2=0.9476
  iters LGB=948 CB=1988
blend alpha=0.4 LGB: RMSE=0.2069
blend alpha=0.5 LGB: RMSE=0.2072
blend alpha=0.6 LGB: RMSE=0.2079
blend alpha=0.7 LGB: RMSE=0.2088
blen

(0.3592959277683009, 0.20942137584954482, 0.8008610339125828)

In [22]:
test = load_bgg('test_data.csv')
test['is_remastered'] = test['Name'].str.contains(NAME_RE, case=False, na=False).astype(int)

n_lgb = int(np.mean(lgbm_best_iters)) if lgbm_best_iters else 2000
n_cb = int(np.mean(cb_best_iters)) if cb_best_iters else 2000
feat = BoardGameFeatures(**feat_kwargs)
X_all = feat.fit_transform(X, y)
Xt = feat.transform(test[features])

lgbm_full = make_lgbm(n_estimators=max(n_lgb, 200), **best_lgbm_params)
lgbm_full.fit(X_all, y)
cb_full = make_catboost(iterations=max(n_cb, 200))
cb_full.fit(X_all, y)

pred = np.clip(
    blend_alpha * lgbm_full.predict(Xt) + (1.0 - blend_alpha) * cb_full.predict(Xt),
    1, 10,
)
submission = pd.DataFrame({
  'index': test.index,
  'Rating Average': pred,
})
submission.to_csv('submission.csv', index=False)
submission.head()


,index,Rating Average
0,0,8.210409
1,1,8.141634
2,2,8.308212
3,3,8.319999
4,4,8.258353


In [ ]:
# Инверсия Bayes average по BGG Rank.
# G = (R*n + C*prior) / (n+C), ранг монотонен по G.
# На тесте G берётся с изотонической кривой ранг→G, обученной только на train, затем
# R = (G*(n+C) - C*prior) / n.

from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

bayes_df = load_bgg('train_data.csv')
y_b = bayes_df['Rating Average'].to_numpy(float)
n_b = bayes_df['Users Rated'].to_numpy(float)
rank_b = bayes_df['BGG Rank'].to_numpy(float)


def _bayes_predict(rank_tr, y_tr, n_tr, rank_va, n_va, C, prior):
    geek = (y_tr * n_tr + C * prior) / (n_tr + C)
    iso = IsotonicRegression(increasing=False, out_of_bounds='clip')
    iso.fit(rank_tr, geek)
    geek_hat = iso.predict(rank_va)
    rating = (geek_hat * (n_va + C) - C * prior) / np.maximum(n_va, 1.0)
    return np.clip(rating, 1, 10)


def _rmse(a, b):
    return float(np.sqrt(mean_squared_error(a, b)))


def _cv_rmse(C, prior, n_splits=3):
    scores = []
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=0)
    for tr_i, va_i in kf.split(y_b):
        pred = _bayes_predict(
            rank_b[tr_i], y_b[tr_i], n_b[tr_i], rank_b[va_i], n_b[va_i], C, prior,
        )
        scores.append(_rmse(y_b[va_i], pred))
    return float(np.mean(scores))


grid = []
for C in (1200, 1600, 2000, 2400, 3000):
    for prior in (5.4, 5.5, 5.55, 5.7):
        grid.append((_cv_rmse(C, prior), C, prior))
grid.sort()
best_cv, best_C, best_prior = grid[0]

# Доподбор вокруг лучшей точки сетки.
for C in np.arange(best_C - 200, best_C + 201, 50):
    for prior in np.round(np.arange(best_prior - 0.1, best_prior + 0.11, 0.05), 2):
        grid.append((_cv_rmse(float(C), float(prior)), float(C), float(prior)))
grid.sort()
best_cv, best_C, best_prior = grid[0]
train_pred = _bayes_predict(rank_b, y_b, n_b, rank_b, n_b, best_C, best_prior)
train_rmse = _rmse(y_b, train_pred)

print(f'best C={best_C:.0f} prior={best_prior:.2f}')
print(f'CV RMSE={best_cv:.4f}  train RMSE={train_rmse:.4f}')
print('топ сетки:')
for cv, C, prior in grid[:5]:
    print(f'  cv={cv:.4f} C={C:.0f} prior={prior:.2f}')

slices = pd.qcut(n_b, 5)
print('RMSE по квантилям Users Rated:')
for interval, idx in bayes_df.groupby(slices, observed=True).groups.items():
    ii = np.asarray(list(idx))
    print(f'  {interval}: RMSE={_rmse(y_b[ii], train_pred[ii]):.4f}  n_med={np.median(n_b[ii]):.0f}')

print(
    'Потолок формулы около 0.24, не 0.1. '
    'Ошибка G ~0.02 умножается на (n+C)/n: при n~40 и C~2000 это ×50. '
    'У игр с сотнями оценок RMSE уже < 0.1, хвост n=30–80 вытягивает среднее.'
)
